In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
%%sql
select * from dbo.bronze_orders

In [ ]:
df_orders_raw = spark.read.parquet("abfss://e8b72d8d-5a0f-40b4-bcfc-e5ff1552d786@onelake.dfs.fabric.microsoft.com/ac40a136-0e9a-48e0-ab5d-652add7f1fed/Files/bronze/orders_data")
display(df_orders_raw)

# Orders Data Clean Up

1. Change Order_date column Data Type

In [ ]:
from pyspark.sql import functions as F

# 1. Clean up ordinal suffixes (like 'st', 'nd', 'rd', 'th') from text dates 
# This changes "July 20th 2023" to "July 20 2023" so Spark can parse it easily
df_cleaned = df_orders_raw.withColumn(
    "cleaned_date", 
    F.regexp_replace(F.col("order_date"), r"(?<=\d)(st|nd|rd|th)", "")
)

# 2. Map known formats in the order of priority. 
# Explicit formats handle ambiguities like whether 12-06-2023 is June 12 or Dec 6.
date_formats = [
    "MM.dd.yyyy",    # 07.21.2023
    "yyyy.MM.dd",    # 2023.07.10
    "MMMM dd yyyy",  # July 20 2023 (after regex cleaning)
    "MM/dd/yyyy",    # 07/15/2023
    "dd-MM-yyyy",    # 12-06-2023, 15-07-2023
    "yyyy/dd/MM",    # 2023/13/07
    "MM-dd-yyyy",    # 07-20-2023
]

# 3. Apply coalesce to attempt each format until one succeeds
df_standardized = df_cleaned.withColumn(
    "standard_order_date",
    F.coalesce(*[F.to_date(F.col("cleaned_date"), fmt) for fmt in date_formats])
).drop("cleaned_date")

#display(df_standardized)
df_order=df_standardized.select('Order_ID','cust_id', 'Product_Name', 'Qty','Order_Amount$','Delivery_Status','Payment_Mode','Ship_Address','Email', 'Promo_Code', 'Feedback_Score', 'standard_order_date')
display(df_order)

2. Rename the required Columns

In [ ]:
df_order=df_order.withColumnRenamed('cust_id', 'Customer_ID').withColumnRenamed('Order_Amount$', 'Order_Amount').withColumnRenamed('standard_order_date', 'Order_Date')
#.withColumnRenamed('OrderAmount', 'Order_Amount')
#df_order=df_order.withColumnRenamed('standard_order_date', 'Order_Date')
display(df_order)

In [ ]:
df_order=df_order.withColumnRenamed('Qty', 'Quantity')
display(df_order)

3. Change Quantity Column Data from String to Integers and Convert all the numbers into Whole Numbers

In [ ]:
# Dictionary to map written-out numbers to integers
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from itertools import chain

text_to_num = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "nan": 0}

# 1. Create the PySpark mapping expression
mapping_expr = F.create_map([F.lit(x) for x in chain(*text_to_num.items())])

# 2. Clean the string column
cleaned_qty = F.lower(F.trim(df_order["Quantity"].cast("string")))

# 3. Use F.coalesce to handle the lookup, the numeric fallback, and the 0 default
df_order = df_order.withColumn(
    "Quantity",
    F.coalesce(
        mapping_expr[cleaned_qty],            # 1st choice: Dictionary lookup ("one" -> 1)
        df_order["Quantity"].cast(IntegerType()),    # 2nd choice: Direct number cast ("3" -> 3)
        F.lit(0)                              # 3rd choice: Default fallback if both fail
    )
)


display(df_order)

4. adding Default Values for Promo Code

In [ ]:
from pyspark.sql.functions import *
df_order=df_order.withColumn("Promo_Code", coalesce(col("Promo_Code"), lit("NO_PROMO")))
display(df_order)

5. Change the Feedback Score for Null values, BAD and NAN etc.

In [ ]:
# Dictionary to map written-out numbers to integers
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from itertools import chain

#Feedback_Score
text_to_num = {"bad":0, "nan": 0}

# 1. Create the PySpark mapping expression
mapping_expr = F.create_map([F.lit(x) for x in chain(*text_to_num.items())])

# 2. Clean the string column
cleaned_qty = F.lower(F.trim(df_order["Quantity"].cast("string")))

# 3. Use F.coalesce to handle the lookup, the numeric fallback, and the 0 default
df_order = df_order.withColumn(
    "Feedback_Score",
    F.coalesce(
        mapping_expr[cleaned_qty],            # 1st choice: Dictionary lookup ("one" -> 1)
        df_order["Feedback_Score"].cast(IntegerType()),    # 2nd choice: Direct number cast ("3" -> 3)
        F.lit(0)                              # 3rd choice: Default fallback if both fail
    )
)


display(df_order)

6. Change Null Values for Delivery_Status and Payment_Mode

In [ ]:
df_order=df_order.withColumn("Delivery_Status", coalesce(col("Delivery_Status"), lit("NA Now"))).withColumn('Payment_Mode', coalesce(col('Payment_Mode'), lit('No Payment Now')))
display(df_order)

7. Add new col Order_Year, and Order_Month using Order_date and Also add Order_Hash Col

In [ ]:
# Add year and month columns for easier reporting
df_order=df_order.withColumn("Order_Year", year("Order_Date")).withColumn("Order_Month", month("Order_Date")).withColumn("Order_Hash", sha2(concat_ws("|", *df_order.columns), 256))

display(df_order)

8. Change Order Amount

In [ ]:
from pyspark.sql import functions as F

# Pattern matches $, usd, rs, inr (case-insensitive via (?i))
currency_pattern = r"(?i)\$|₹|usd|rs|inr"

df_order = df_order.withColumn(
    "Order_Amount", 
    F.trim(F.regexp_replace(F.col("Order_Amount"), currency_pattern, ""))
)

display(df_order)

In [ ]:
df_order = df_order.withColumn(
    "Order_Amount", 
    F.regexp_replace(F.col("Order_Amount"), r"\.", "")
)
display(df_order)

In [ ]:
#Using IntegerType (requires import)
from pyspark.sql.types import IntegerType
df_order = df_order.withColumn("Order_Amount", df_order["Order_Amount"].cast(IntegerType()))
display(df_order)

In [ ]:
df_order=df_order.withColumn("Email", lower(trim(col("Email"))))
display(df_order)

9. Drop duplicate columns

In [ ]:
# Drop rows where critical fields are missing
df_order=df_order.dropna(subset=["Order_ID", "Order_Amount", "Customer_ID"])

# Remove duplicate Order IDs
df_order=df_order.dropDuplicates(["Order_ID"])
display(df_order)

In [ ]:
# Save cleaned Orders data
df_order.write.mode("overwrite").format("delta").saveAsTable("silver_orders")